In [4]:
import requests
import numpy as np
import pandas as pd
from datetime import datetime, timezone
from pathlib import Path
import os
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

import database

In [5]:
API_KEY = os.environ['KEY']

In [6]:
FINAL_SERIES_IDS = ['UNRATE', 'INDPRO', 'JTSJOL']
feature_set = ['log10_INDPRO', 'log10_JTSJOL', 'log10_UNRATE', 'UNRATE']

# Get date of last update on FRED

In [7]:
with requests.Session() as session:

    url = "https://api.stlouisfed.org/fred/series"
    last_update_fred = {}

    for ID in FINAL_SERIES_IDS:
        params = {"api_key": API_KEY,
                  "file_type": "json",
                  "series_id": ID}

        r = session.get(url=url, params=params, timeout=30)
        r.raise_for_status()

        data = r.json()

        fred_local_time = datetime.strptime(data['seriess'][0]['last_updated'] + "00", "%Y-%m-%d %H:%M:%S%z")

        last_update_fred.update({ID: fred_local_time.strftime("%Y-%m-%d %H:%M:%S")})

    last_update_fred = {
    series_id: datetime.fromisoformat(date)
    for series_id, date in last_update_fred.items()
    }


In [8]:
last_update_fred

{'UNRATE': datetime.datetime(2026, 8, 7, 8, 31, 41),
 'INDPRO': datetime.datetime(2026, 8, 18, 8, 20, 15),
 'JTSJOL': datetime.datetime(2026, 8, 4, 9, 10, 15)}

In [9]:
type(last_update_fred['UNRATE'])

datetime.datetime

# Get date of latest feature entry from db
Table: `FREDSeries`; Column: `FRED_LastUpdateLocalTime`

In [10]:
last_update_db = {}
db = database.get_fred_last_update_local(series_ids=FINAL_SERIES_IDS)
last_update_db = {
    series_id: datetime.fromisoformat(date)
    for series_id, date in db
}

In [11]:
last_update_db

{'INDPRO': datetime.datetime(2026, 7, 17, 8, 20, 24),
 'JTSJOL': datetime.datetime(2026, 8, 4, 9, 10, 15),
 'UNRATE': datetime.datetime(2026, 8, 7, 8, 31, 41)}

In [12]:
type(last_update_db['UNRATE'])

datetime.datetime

# Compare dates

In [13]:
# last_update_fred = {'UNRATE': datetime(2026, 9, 7, 8, 31, 41),
#  'INDPRO': datetime(2026, 9, 18, 8, 20, 15),
#  'JTSJOL': datetime(2026, 9, 4, 9, 10, 15)}

In [14]:
last_update_fred

{'UNRATE': datetime.datetime(2026, 8, 7, 8, 31, 41),
 'INDPRO': datetime.datetime(2026, 8, 18, 8, 20, 15),
 'JTSJOL': datetime.datetime(2026, 8, 4, 9, 10, 15)}

In [15]:
all_db_series_outdated = all(
    ID not in last_update_db
     or last_update_fred[ID] > last_update_db[ID]
    for ID in last_update_db
)
all_db_series_outdated

False

# Final code

In [16]:
# --- returns true if all entries of the final selection if series IDs is outdated ---

def check_db_all_entries_outdated(final_series_ids, feature_set):
    # get last update date of series IDs, which are part of the feature set, from FRED
    with requests.Session() as session:

        url = "https://api.stlouisfed.org/fred/series"
        last_update_fred = {}

        for ID in final_series_ids:
            params = {"api_key": API_KEY,
                    "file_type": "json",
                    "series_id": ID}

            r = session.get(url=url, params=params, timeout=30)
            r.raise_for_status()

            data = r.json()

            fred_local_time = datetime.strptime(data['seriess'][0]['last_updated'] + "00", "%Y-%m-%d %H:%M:%S%z")

            last_update_fred.update({ID: fred_local_time.strftime("%Y-%m-%d %H:%M:%S")})

        last_update_fred = {
        series_id: datetime.fromisoformat(date)
        for series_id, date in last_update_fred.items()
        }

    # get last update date of the final IDs from the feature set from the db
    last_update_db = {}
    db = database.get_fred_last_update_local(series_ids=final_series_ids)
    last_update_db = {
        series_id: datetime.fromisoformat(date)
        for series_id, date in db
    }

    all_db_series_outdated = all(
        ID not in last_update_db
        or last_update_fred[ID] > last_update_db[ID]
        for ID in last_update_db
    )

    return all_db_series_outdated

In [19]:
check_db_all_entries_outdated(FINAL_SERIES_IDS, feature_set)

True